# 🚀 MyBabyAI Cloud Trainer

**Colab + Kaggle Uyumlu** — Ortamı otomatik algılar.

| Platform | GPU | Süre/Hafta | Disk |
|---|---|---|---|
| Colab Free | T4 (15 GB) | ~4-5 saat | ~100 GB |
| Colab Pro | T4/A100 | ~24 saat | ~200 GB |
| Kaggle | T4/P100 (16 GB) | 30 saat | ~70 GB |

> **Öneri:** Kaggle 30 saat/hafta veriyor, Colab free saatler içinde keser. **Uzun eğitimler için Kaggle tercih edin.**

In [ ]:
# ══════════════════════════════════════════════════
# 1. ORTAM TESPITI (Colab / Kaggle / Yerel)
# ══════════════════════════════════════════════════
import os, sys

ENV = "local"
if os.path.exists("/content"):
    ENV = "colab"
elif os.path.exists("/kaggle"):
    ENV = "kaggle"

print(f"📡 Ortam: {ENV.upper()}")
print(f"📂 Çalışma dizini: {os.getcwd()}")

# GPU kontrolü
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"🎮 GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("⚠️ GPU bulunamadı! Eğitim çok yavaş olacak.")
    print("   Kaggle: Settings > Accelerator > GPU T4 x2 seçin")
    print("   Colab: Runtime > Change runtime type > T4 GPU seçin")

In [ ]:
# ══════════════════════════════════════════════════
# 2. BAĞIMLILIKLAR + HF TOKEN
# ══════════════════════════════════════════════════
!pip install -q torch torchvision transformers accelerate peft bitsandbytes \
    sentencepiece tokenizers safetensors huggingface-hub chromadb \
    sentence-transformers PyYAML python-dotenv rich tqdm numpy pandas \
    pillow datasets requests beautifulsoup4 psutil

# HF Token (isteğe bağlı — gated dataset'ler için gerekli)
hf_token = None

if ENV == "colab":
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        print("✅ Colab HF Token yüklendi.")
    except Exception:
        print("ℹ️ Colab'da HF Token bulunamadı (isteğe bağlı).")

elif ENV == "kaggle":
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        hf_token = secrets.get_secret("HF_TOKEN")
        print("✅ Kaggle HF Token yüklendi.")
    except Exception:
        print("ℹ️ Kaggle'da HF Token bulunamadı (isteğe bağlı).")
        print("   Eklemek için: Add-ons > Secrets > 'HF_TOKEN' adıyla ekleyin")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token

In [ ]:
# ══════════════════════════════════════════════════
# 3. REPO KLONLAMA
# ══════════════════════════════════════════════════
REPO_URL = "https://github.com/halilibrahimavsar/mybabyai.git"
REPO_DIR = "mybabyai"

if os.path.exists(".git") or REPO_DIR in os.getcwd():
    print("✅ Zaten repo dizinindeyiz, güncelleniyor...")
    !git fetch --all
    !git reset --hard origin/main || git reset --hard origin/master
    !git pull
elif os.path.exists(REPO_DIR):
    print(f"✅ '{REPO_DIR}' bulundu, güncelleniyor...")
    %cd {REPO_DIR}
    !git fetch --all
    !git reset --hard origin/main || git reset --hard origin/master
    !git pull
else:
    print("⬇️ Repo klonlanıyor...")
    !git clone {REPO_URL}
    %cd {REPO_DIR}

sys.path.insert(0, os.getcwd())
print(f"✅ Çalışma dizini: {os.getcwd()}")

---
## ⚙️ Kontrol Paneli
Aşağıdaki ayarları değiştirerek eğitimi yapılandırın:

In [ ]:
# @title 🚀 Eğitim Kontrol Paneli
# @markdown Eğitim türü, model boyutu ve checkpoint ayarları

training_mode = "full" # @param ["full", "lora"]
model_size = "350M-MoE" # @param ["125M", "350M", "350M-MoE", "650M"]
load_from_checkpoint = False # @param {type:"boolean"}
num_epochs = 3 # @param {type:"integer"}
batch_size = 4 # @param {type:"integer"}
gradient_accumulation = 4 # @param {type:"integer"}
learning_rate = 3e-4 # @param {type:"number"}
save_steps = 100 # @param {type:"integer"}

# LoRA için düşük LR önerilir
if training_mode == "lora":
    learning_rate = min(learning_rate, 1e-4)

# Kaggle'da checkpoint çıktı yolu
output_dir = "/kaggle/working/checkpoints" if ENV == "kaggle" else "./checkpoints"
os.makedirs(output_dir, exist_ok=True)

print(f"""
╔════════════════════════════════════════╗
║  MyBabyAI Eğitim Yapılandırması         ║
╠════════════════════════════════════════╣
║  Ortam:      {ENV.upper():>25} ║
║  Mod:        {training_mode.upper():>25} ║
║  Model:      CodeMind-{model_size:>16} ║
║  Checkpoint: {'Evet' if load_from_checkpoint else 'Hayır (Sıfırdan)':>25} ║
║  Epoch:      {num_epochs:>25} ║
║  Batch:      {batch_size:>25} ║
║  Grad Acc:   {gradient_accumulation:>25} ║
║  LR:         {learning_rate:>25} ║
║  Save Her:   {save_steps:>25} adım ║
║  Çıktı:      {output_dir:>25} ║
╚════════════════════════════════════════╝
""")

In [ ]:
# ══════════════════════════════════════════════════
# 4. MODEL & TRAINER BASLATMA
# ══════════════════════════════════════════════════
from src.core.model_manager import ModelManager
from src.core.trainer import LoRATrainer
from src.utils.config import Config

config = Config()

# Kaggle'da checkpoint yolunu config'e yaz
if ENV == "kaggle":
    config.set("training.output_dir", output_dir)
    config.set("training.checkpoint_dir", output_dir)

model_manager = ModelManager(config)

if load_from_checkpoint:
    print("📦 Mevcut checkpoint yükleniyor...")
    model_manager.load_model("codemind", allow_fresh_fallback=True)
else:
    print(f"🌱 Sıfırdan CodeMind-{model_size} oluşturuluyor...")
    model_manager.load_fresh_model(size=model_size)

trainer = LoRATrainer(model_manager, config)

# Model bilgilerini yazdır
total_params = sum(p.numel() for p in model_manager.model.parameters())
trainable_params = sum(p.numel() for p in model_manager.model.parameters() if p.requires_grad)
print(f"""
✅ Model hazır!
   Ad: {model_manager.model_name}
   Toplam parametre: {total_params:,}
   Eğitilebilir:     {trainable_params:,}
   Cihaz: {model_manager.device}
""")

In [ ]:
# ══════════════════════════════════════════════════
# 5. 🎯 DATASET HAVUZU (Multi-Dataset Training)
# ══════════════════════════════════════════════════
# Aşağıdaki dataset havuzunu ihtiyacınıza göre düzenleyin.
# max_samples değerlerini artırarak daha fazla veri kullanabilirsiniz.
#
# Desteklenen tipler:
#   "huggingface" → HuggingFace dataset (dataset_key ile)
#   "file"        → Yerel dosya (path ile)
#   "url"         → Web sayfasından metin çekme

dataset_pool = [
    {
        "name": "🇬🇧 English Chat (UltraChat)",
        "type": "huggingface",
        "dataset_key": "ultrachat_200k",
        "max_samples": 15000
    },
    {
        "name": "🤖 GPT-4 Instructions (SlimOrca)",
        "type": "huggingface",
        "dataset_key": "slim_orca",
        "max_samples": 15000
    },
    {
        "name": "🇹🇷 Turkish Instructions (Merve)",
        "type": "huggingface",
        "dataset_key": "turkish_instructions_merve",
        "max_samples": 10000
    },
    {
        "name": "🇹🇷 Turkish Alpaca",
        "type": "huggingface",
        "dataset_key": "turkish_alpaca",
        "max_samples": 8000
    },
    {
        "name": "💻 Coding (CodeAlpaca)",
        "type": "huggingface",
        "dataset_key": "code_alpaca_20k",
        "max_samples": 8000
    }
]

total_samples = sum(d.get('max_samples', 0) for d in dataset_pool)
print(f"📊 Dataset havuzu: {len(dataset_pool)} kaynak, ~{total_samples:,} örnek")
for d in dataset_pool:
    print(f"   • {d['name']}: {d.get('max_samples', '?')} örnek")

In [ ]:
# ══════════════════════════════════════════════════
# 6. 🚀 EğİTİM BAŞLA!
# ══════════════════════════════════════════════════
import time

print(f"⏱️ Eğitim başlıyor... ({training_mode.upper()} mod)")
start_time = time.time()

try:
    metrics = trainer.train_from_pool(
        dataset_pool,
        training_type=training_mode,
        use_notebook_callback=True,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation,
        learning_rate=learning_rate,
        weight_decay=0.01,
        logging_steps=10,
        save_steps=save_steps,
        output_dir=output_dir,
    )
    
    elapsed = time.time() - start_time
    hours, remainder = divmod(elapsed, 3600)
    minutes, seconds = divmod(remainder, 60)
    
    print(f"""
    ════════════════════════════════════════
    ✅ Eğitim Tamamlandı!
    ════════════════════════════════════════
    Süre: {int(hours)}s {int(minutes)}dk {int(seconds)}sn
    Metrics: {metrics}
    Checkpoint: {output_dir}
    """)
    
except Exception as e:
    elapsed = time.time() - start_time
    print(f"❌ Eğitim hatası ({elapsed:.0f}sn sonra): {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ══════════════════════════════════════════════════
# 7. CHECKPOINT İNDİRME (Kaggle için önemli!)
# ══════════════════════════════════════════════════
# Kaggle'da dosyaları indirmeniz gerekir çünkü session bitince silinir.
# Colab'da Drive'a kaydedebilirsiniz.

import shutil, glob

checkpoint_files = glob.glob(f"{output_dir}/**/*.pt", recursive=True)
checkpoint_files += glob.glob(f"{output_dir}/**/*.safetensors", recursive=True)

if checkpoint_files:
    print(f"📦 {len(checkpoint_files)} checkpoint dosyası bulundu:")
    for f in checkpoint_files:
        size_mb = os.path.getsize(f) / 1e6
        print(f"   • {os.path.basename(f)} ({size_mb:.1f} MB)")
else:
    # Model final checkpoint
    codemind_path = os.path.join(os.getcwd(), "models", "codemind")
    checkpoint_files = glob.glob(f"{codemind_path}/**/*.pt", recursive=True)
    if checkpoint_files:
        print(f"📦 {len(checkpoint_files)} checkpoint (models/codemind):")
        for f in checkpoint_files:
            size_mb = os.path.getsize(f) / 1e6
            print(f"   • {f} ({size_mb:.1f} MB)")

if ENV == "colab":
    print("\n💾 Colab → Google Drive'a kaydetmek için:")
    print("   from google.colab import drive")
    print("   drive.mount('/content/drive')")
    print(f"   !cp -r {output_dir} /content/drive/MyDrive/mybabyai_checkpoints/")
elif ENV == "kaggle":
    print("\n💾 Kaggle → Output dosyaları otomatik kaydedilir:")
    print("   Sağ üstte 'Save Version' → 'Save & Run All'")
    print("   Sonra: Output sekmesinden indirin")
    # /kaggle/working/ altındaki dosyalar otomatik output olur
    if output_dir != "/kaggle/working/checkpoints":
        dst = "/kaggle/working/checkpoints"
        if os.path.exists(output_dir):
            shutil.copytree(output_dir, dst, dirs_exist_ok=True)
            print(f"   ✅ Checkpoint'ler {dst}'a kopyalandı")

In [ ]:
# ══════════════════════════════════════════════════
# 8. 🧪 TEST - Eğitilmiş modeli dene
# ══════════════════════════════════════════════════
from src.core.inference import InferenceEngine

engine = InferenceEngine(model_manager, config=config)

test_prompts = [
    "Merhaba, nasılsın?",
    "Python'da fibonacci fonksiyonu yaz.",
    "Türkiye'nin başkenti neresidir?",
]

print("─" * 50)
for prompt in test_prompts:
    print(f"\n🗣️ Soru: {prompt}")
    try:
        response = engine.generate(prompt, use_memory=False, max_new_tokens=128)
        print(f"🤖 Cevap: {response[:300]}")
    except Exception as e:
        print(f"❌ Hata: {e}")
    print("─" * 50)